# AI Video Comparison Tool — Kaggle

Setup:
1. GPU: **Settings → Accelerator → GPU T4 x2** (or P100)
2. Internet: **ON** (for pip install + vieneu)
3. Upload các asset cố định vào `assets/`:
   - `background.jpg` — nền video
   - `character.png` — nhân vật tường thuật chính
   - `character_confused.png` — nhân vật tường thuật (băn khoăn)
   - `image_a.jpg` — ảnh nhân vật A
   - `image_b.jpg` — ảnh nhân vật B
   - `ref_voice.wav` — file giọng mẫu (15-30s) để clone giọng custom

Không cần LLM/GGUF ở đâu cả — kịch bản do AI ngoài (ChatGPT/Gemini) viết, notebook chỉ render.

In [ ]:
# Cell 1: Clone repo + set working directory
import os

REPO_DIR = '/kaggle/working/ai-video-comparison-tool'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vinhff-ff/ai-video-comparison-tool.git {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 2: Cài deps nhẹ (không có LLM — chỉ TTS vieneu, playwright, ffmpeg)
!pip install -q playwright ffmpeg-python
!playwright install chromium

from vieneu import Vieneu  # sẽ chạy trong venv riêng ở cell dưới nếu cần

In [ ]:
# Cell 3: Nhập kịch bản (mảng 10-15 câu thoại do AI ngoài viết)
import json

SCRIPT_LINES = [
    'Đây là nghi can A.',
    'Đây là nghi can B.',
    'Vậy hai kẻ này có tội khác nhau thế nào.',
    'A lừa đảo chiếm đoạt tài sản, lĩnh 5 năm tù.',
    'B buôn lậu qua biên giới, lĩnh 12 năm tù.',
    'Hành vi của A tinh vi, che giấu qua nhiều công đoạn.',
    'Hành vi của B liều lĩnh, vận chuyển số lượng lớn.',
    'Cả hai đều phải trả giá cho lựa chọn sai lầm.',
    'Bài học đặt ra: ranh giới mong manh giữa giàu nhanh và phạm tội.',
    'Hãy sống đúng pháp luật, đừng học theo họ.',
]
print(len(SCRIPT_LINES), 'lines')

In [ ]:
# Cell 4: Run pipeline — vieneu với giọng custom (ref_audio)
import sys; sys.path.insert(0, 'src')
from pipeline import generate_video_phase3

assets = {
    'background': 'assets/background.jpg',
    'character':  'assets/character.png',
    'character_confused': 'assets/character_confused.png',
    'image_a':    'assets/image_a.jpg',
    'image_b':    'assets/image_b.jpg',
}

result = await generate_video_phase3(
    script_lines=SCRIPT_LINES,
    assets=assets,
    run_id='clone_run_001',
    engine='vieneu',
    ref_audio='assets/ref_voice.wav',
)
print('DONE:', result)

In [ ]:
# Cell 5: Xem kịch bản đã map (generated/scripts/<run_id>.json)
import json
d = json.load(open('generated/scripts/clone_run_001.json'))
for i, s in enumerate(d['scenes']):
    print(f'[{i}] {s["text"]}\n     anim={s["animation"]} image={s["image"]} char={s["character"]}\n')

In [ ]:
# Cell 6: Download kết quả
from pathlib import Path

for mp4 in Path('generated/videos').glob('*_final.mp4'):
    print(f'  → {mp4.name}  ({mp4.stat().st_size / 1024:.0f} KB)')

# Click phải vào file trong Output panel → Download